# 02 — Finance & Safety Agents
**Agentic X | Studio 67** — Real components only. Run from the repository root.

## 1. Objective
Two independent Python agents. Finance applies monetary constraints; Safety has a hard veto. Both return the same `AgentEvaluation` model as Creative.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'core').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from core.environment import get_case
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
finance=FinanceAgent(); safety=SafetyAgent()

## 2. Real screen-failure case
Data comes from the dataset, not hard-coded mock responses.

In [ ]:
case, options = get_case('SCR-001')
for option in options:
    fin=finance.evaluate(case,option)
    saf=safety.evaluate(case,option)
    print(option['option_id'],'FINANCE',fin.approved,fin.reason,'SAFETY',saf.approved,saf.reason)

## 3. Budget-overrun scenario
For Budget Overrun, the cost after action equals projected spending plus option cost minus verified gross savings.

In [ ]:
budget_case, budget_options = get_case('BDG-001')
print('Projected spending:',budget_case['projected_total_spend_sar'],'Budget:',budget_case['initial_budget_sar'])
for option in budget_options:
    print(option['option_id'], finance.evaluate(budget_case,option).model_dump())

## 4. Safety veto regression tests
These are constructed input **test cases**, not Mock Agents. They verify that a veto blocks unsafe decisions.

In [ ]:
unsafe={'option_id':'TEST-UNSAFE','safety_ok':False,'permit_ok':True,'mitigation_verified':True,'risk_score_after_mitigation':1}
assert safety.evaluate({},unsafe).approved is False
expensive={'option_id':'TEST-COST','cost_sar':999999}
assert finance.evaluate({'scenario':'Screen Failure','initial_budget_sar':5000,'committed_spend_sar':0},expensive).approved is False
print('Budget and safety blocking tests passed')

## 5. Combined structured result
Notebook 04 can consume both agents with their real `evaluate(case,option)` method.

In [ ]:
evaluations=[agent.evaluate(case,options[0]) for agent in [finance,safety]]
assert all(r.option_id==options[0]['option_id'] for r in evaluations)
print([x.model_dump() for x in evaluations])